# Data Splitting Strategy

This notebook implements the data splitting strategy for the fraud detection model, including:
1. Out-of-time (OOT) split
2. Train/validation split
3. Sampling approaches

## Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import sys
sys.path.append('../../')
from utils.multicolumn_encoder import MultiColumnEncoder

# Load processed data
data = pd.read_parquet('../../data/processed_credit_card_transactions.parquet')

## set transaction num as index

In [2]:
data.set_index('trans_num', inplace=True)

## Out-of-Time Split

Split data into training and out-of-time test sets based on transaction date.

In [3]:
_expression = "trans_date_trans_time < '2020-07-01 00:00:00'"
print(f"Splitting dataframe based on expression {_expression!r}.")

# Split into training and OOT sets
train = data.query(_expression)
oot = data.query(f"~({_expression})")

print(f"Split dataframe into two dataframes with shapes {train.shape} and {oot.shape}.")

# Separate features and target for OOT set
oot_y = oot.is_fraud
oot_X = oot.drop(columns=['is_fraud'])

Splitting dataframe based on expression "trans_date_trans_time < '2020-07-01 00:00:00'".
Split dataframe into two dataframes with shapes (1326733, 18) and (525661, 18).


## Train/Validation Split

In [4]:
# Prepare training data
X = train.drop(columns=['is_fraud'])
y = train['is_fraud']

# Split into train and holdout sets
train_X, holdout_X, train_y, holdout_y = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y
)

## Transform Features

In [5]:
categorical_columns = X.select_dtypes(include=['object']).columns.tolist()
encoder = MultiColumnEncoder(categorical_columns)
train_X = encoder.fit_transform(train_X)
holdout_X = encoder.transform(holdout_X)
oot_X = encoder.transform(oot_X)

## Save Split Datasets

In [6]:
# Save the different splits
train_data = pd.concat([train_X, train_y], axis=1)
holdout_data = pd.concat([holdout_X, holdout_y], axis=1)
oot_data = pd.concat([oot_X, oot_y], axis=1)

train_data.to_parquet('../../data/train_data.parquet')
holdout_data.to_parquet('../../data/holdout_data.parquet')
oot_data.to_parquet('../../data/oot_data.parquet')